# 46 · Space-time scan, checked against numpy

## What this checks

**Space-time scan** looks for places where cases came closer together in time than the rest of the outbreak would
lead us to expect. This check covers release 0.8.0.

It uses the space-time permutation scan (Kulldorff and others, 2005). The scan draws circles around places and
windows of days. For each one, it compares the cases seen with the cases expected if timing had nothing to do with
place. It can run once over the whole outbreak, or day by day as the outbreak unfolded. This check covers both.

A cluster flagged too early, too late, or in the wrong place changes where a team goes. So it is worth checking.

## How it is checked

We write the same scan again from its published definition, using **numpy**, Python's library for arrays of
numbers. No code is shared with Fieldwork.

The two sides differ in two ways, on purpose:

- **Every run of days.** numpy scores every run of days. Fieldwork, to save time, tries only runs that start and end
  on a day with a case. So the check also tests that shortcut.
- **Own random numbers.** numpy shuffles the dates with its own random numbers. So the p-values cannot share
  Fieldwork's random stream.

The data are the measles outbreak in Hagelloch, Germany, in 1861. There are 188 cases at 56 places, by day of the
first symptoms (prodrome), placed on the houses of the village's 1824 map. Both sides use 999 shuffles. Run in
CPython, the shuffles take about a minute; in the browser, about three.

## What should happen

1. **The same scan.** The same number of circles, runs of days and windows.
2. **The same clusters.** For each cluster, the same places, days, cases seen, cases expected and score, to 9
   decimal places.
3. **Close p-values.** A p-value from shuffles is an estimate, so two independent ones differ a little. Each must be
   within three standard errors of the difference between two estimates, plus the smallest step a p-value can take
   with 999 shuffles.
4. **The same day by day.** On every day, the same decision to scan or skip, and the same most likely cluster. Its
   p-value must be within the same margin on every day Fieldwork's is 0.1 or less. Those are the days that decide
   when the scan first flags.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_46_space_time_scan import run

report = run(Path.cwd() / "fixtures" / "scan-hagelloch-001.json")
print("agrees:", report["agrees"])
print(report["totals"])
m = report["mostLikely"]
print(f"most likely: {m['places']} place, {m['start']} to {m['end']}, {m['observed']} observed, {m['expected']} expected, p = {m['pValue']}")
for p in report["pValues"]:
    print(f"  cluster {p['rank']:2}: p {p['fieldwork']:.3f} in Fieldwork, {p['here']:.3f} here, allowed {p['allowed']:.3f}")
print("differs:", report["differs"])
r = report["replay"]
print(f"replay: first at or under 0.05 on {r['firstAt05InFieldwork']} in Fieldwork, {r['firstAt05Here']} here")
for p in r["pValuesChecked"][:4]:
    print(f"  {p['day']}: p {p['fieldwork']:.3f} in Fieldwork, {p['here']:.3f} here")
print("replay differs:", r["differs"])

## The result on record

First recorded on 14 November 2026, against release 0.6.0. Rerun on 26 November 2026, against release 0.8.0:
**Agrees**.

- **The scan.** Both sides scanned 1,305 circles and 2,838 runs of days.
- **The clusters.** They found the same ten clusters, each with the same places, days, counts and score.
- **The most likely cluster.** It is one house: 8 cases where 0.5532 were expected, from 30 October to 18 November
  1861. Its p-value is 0.001 on both sides.
- **Day by day.** All 87 days came out alike. Both sides first flagged a cluster at p 0.05 or less on 21 November
  1861.

The check found a defect before it agreed. Fieldwork first scanned 1,346 circles, because it measured distances from
exact coordinates but identified places by rounded ones. That was fixed in release 0.1.1.

Fieldwork has since changed how it replays day by day. It stops a day's shuffles once 50 have scored as high as that
day's cluster (Besag and Clifford, 1991), which saves time. The check was run again after that change, and still
agreed.

The result agrees in CPython and in the browser kernel.

Running the cell above repeats the check on the same data.

## What this does not show

- That a cluster is an outbreak. Both sides compute the same statistic. Neither says what it means.
- That this model suits measles in a village. It compares timing across places, and cannot see where people live.
- That the p-values are the same to the digit. They come from different random numbers, so they agree only within
  that margin. On a day near 0.05, the first day flagged could differ for that reason alone.
- The day-by-day p-values where Fieldwork's is above 0.1. Those days are not reshuffled here, to keep the check to
  minutes in a browser. Their clusters and scores are still checked.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("numpy   :", e["numpy"])
print("shuffles:", report["shuffles"])
print("runtime :", e["runtime"], "· Pyodide" if e["pyodide"] else "")